# PPE detection training

These cells are the Kaggle training run, copied back here. They train YOLOv8s and RF-DETR Small on the cleaned 7-class PPE-Normalized-v2 set.

The dataset is found from the working directory: `data/processed/PPE-Normalized-v2`, or a folder named `PPE-Normalized-v2`. Run the eight sections in order.

- YOLOv8 run: `artifacts/yolov8/ppe_yolov8s`
- Copied YOLOv8 weights: `artifacts/weights/yolov8/best.pt`
- RF-DETR run: `artifacts/rfdetr/ppe_rfdetr_small`


## 1. Check GPU

Prints the Python version, the torch version, and whether CUDA is available. When a GPU is present it prints the device name and VRAM. `DEVICE` is `0` on CUDA and `cpu` otherwise. `RF_DEVICE` is `cuda` or `cpu`.


In [1]:
import sys
import torch

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
RF_DEVICE = "cuda" if CUDA else "cpu"

print("Python:", sys.version.split()[0])
print("torch: ", torch.__version__)
print("CUDA:  ", CUDA)
print("device:", DEVICE)
if CUDA:
    print("GPU:   ", torch.cuda.get_device_name(0))
    print("VRAM:  ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("No CUDA GPU. Training will run on CPU.")


Python: 3.13.7
torch:  2.6.0+cu124
CUDA:   True
device: 0
GPU:    NVIDIA GeForce MX130
VRAM:   2.0 GB


## 2. Full PPE-Normalized-v2 class check

Finds the dataset folder, then checks that `data.yaml` lists the seven classes in order: person, helmet, gloves, high_visibility, boots, safety_glasses, harness. Every image must have a label, and every box must be a normalized class id in `0..6`. The cell prints the image count and a box-count table.


In [2]:
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image as ShowImage, display

CLASSES = [
    "person",
    "helmet",
    "gloves",
    "high_visibility",
    "boots",
    "safety_glasses",
    "harness",
]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42


def find_dataset() -> Path:
    relatives = (
        Path("data/processed/PPE-Normalized-v2"),
        Path("PPE-Normalized-v2"),
    )
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        for relative in relatives:
            found = candidate / relative
            if (found / "data.yaml").is_file() and (found / "images").is_dir():
                return found.resolve()
    raise FileNotFoundError(
        "Could not find data/processed/PPE-Normalized-v2 or a PPE-Normalized-v2 folder "
        "above the working directory. Copy the dataset onto this machine and run the "
        "notebook from the PPE-SofEng repo."
    )


DATASET_DIR = find_dataset()
if DATASET_DIR.parent.name == "processed" and DATASET_DIR.parent.parent.name == "data":
    PROJECT_ROOT = DATASET_DIR.parents[2]
else:
    PROJECT_ROOT = DATASET_DIR.parent

print("Dataset:", DATASET_DIR)
print("Project:", PROJECT_ROOT)


def image_files(directory: Path) -> list[Path]:
    if not directory.is_dir():
        return []
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def label_files(directory: Path) -> list[Path]:
    if not directory.is_dir():
        return []
    return sorted(path for path in directory.glob("*.txt") if path.is_file())


def split_names() -> list[str]:
    root = DATASET_DIR / "images"
    return sorted(path.name for path in root.iterdir() if path.is_dir() and not path.name.startswith("_"))


def class_ids(label: Path) -> list[int]:
    found = []
    for line_number, raw in enumerate(label.read_text(encoding="utf-8").splitlines(), start=1):
        fields = raw.split()
        if not fields:
            continue
        class_id = int(float(fields[0]))
        box = [float(value) for value in fields[1:5]]
        if class_id < 0 or class_id >= len(CLASSES):
            raise ValueError(f"class {class_id}: {label}:{line_number}")
        if len(fields) < 5 or any(value < 0.0 or value > 1.0 for value in box):
            raise ValueError(f"bad box: {label}:{line_number}")
        if box[2] <= 0 or box[3] <= 0:
            raise ValueError(f"empty box: {label}:{line_number}")
        found.append(class_id)
    return found


instances = {split: Counter() for split in split_names()}
image_counts = {}
for split in split_names():
    images = image_files(DATASET_DIR / "images" / split)
    labels = label_files(DATASET_DIR / "labels" / split)
    image_stems = {path.stem for path in images}
    label_stems = {path.stem for path in labels}
    missing = image_stems - label_stems
    orphan = label_stems - image_stems
    if missing or orphan:
        raise AssertionError(
            f"{split}: {len(missing)} images without labels, {len(orphan)} orphan labels"
        )
    image_counts[split] = len(images)
    for label in labels:
        for class_id in class_ids(label):
            instances[split][class_id] += 1

names = yaml.safe_load((DATASET_DIR / "data.yaml").read_text(encoding="utf-8"))["names"]
if isinstance(names, dict):
    yaml_names = [names[key] for key in sorted(names, key=lambda item: int(item))]
else:
    yaml_names = list(names)
if yaml_names != CLASSES:
    raise AssertionError(f"data.yaml classes {yaml_names} != {CLASSES}")

frame = pd.DataFrame(
    {split: [instances[split][index] for index in range(len(CLASSES))] for split in instances},
    index=CLASSES,
)
frame["total"] = frame.sum(axis=1)
frame.loc["images"] = {**image_counts, "total": sum(image_counts.values())}
print("[PASS] 7 classes, paired images and labels, boxes in range.")
print("Images:", sum(image_counts.values()))
display(frame.astype(int))


Dataset: C:\Users\My Pc\Documents\PPE-SofEng\data\processed\PPE-Normalized-v2
Project: C:\Users\My Pc\Documents\PPE-SofEng
[PASS] 7 classes, paired images and labels, boxes in range.
Images: 8818


,train,val,total
person,7690,772,8462
helmet,5589,564,6153
gloves,7598,850,8448
high_visibility,5628,565,6193
boots,5562,555,6117
safety_glasses,1635,174,1809
harness,1077,122,1199
images,7936,882,8818


## 3. 90 train / 10 val split

Pools the current train, val, and test images, then keeps 90% for training and 10% for validation. Harness images and safety-glasses images are split inside their own groups so validation still contains those classes. Seed is 42.

If `split_90_10.json` is already in the dataset folder, the cell prints that split and does not move files again. The Kaggle training used the split this cell writes.


In [3]:
SPLIT_MARKER = DATASET_DIR / "split_90_10.json"
HARNESS_ID = CLASSES.index("harness")
GLASSES_ID = CLASSES.index("safety_glasses")


def stratum_of(label: Path) -> str:
    present = set(class_ids(label))
    if HARNESS_ID in present:
        return "harness"
    if GLASSES_ID in present:
        return "safety_glasses"
    return "other"


def pairs_in(split: str) -> list[tuple[Path, Path]]:
    found = []
    for image in image_files(DATASET_DIR / "images" / split):
        label = DATASET_DIR / "labels" / split / f"{image.stem}.txt"
        if not label.is_file():
            raise FileNotFoundError(label)
        found.append((image, label))
    return found


if SPLIT_MARKER.is_file():
    record = json.loads(SPLIT_MARKER.read_text(encoding="utf-8"))
    print("Split already written:", record["train_images"], "train,", record["val_images"], "val")
else:
    grouped = defaultdict(list)
    for split in split_names():
        for image, label in pairs_in(split):
            grouped[stratum_of(label)].append((image, label))

    rng = random.Random(SEED)
    train_pairs = []
    val_pairs = []
    for name, items in sorted(grouped.items()):
        items = sorted(items, key=lambda pair: pair[0].name)
        rng.shuffle(items)
        val_count = max(1, round(len(items) * 0.10)) if len(items) > 1 else 0
        val_pairs.extend(items[:val_count])
        train_pairs.extend(items[val_count:])
        print(f"{name}: {len(items) - val_count} train, {val_count} val")

    staging = DATASET_DIR / "_split90"
    if staging.exists():
        shutil.rmtree(staging)
    for bucket, pairs in (("train", train_pairs), ("val", val_pairs)):
        for image, label in pairs:
            image_dest = staging / "images" / bucket / image.name
            label_dest = staging / "labels" / bucket / label.name
            image_dest.parent.mkdir(parents=True, exist_ok=True)
            label_dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(image), image_dest)
            shutil.move(str(label), label_dest)

    for kind in ("images", "labels"):
        folder = DATASET_DIR / kind
        if folder.exists():
            shutil.rmtree(folder)
        shutil.move(str(staging / kind), folder)
    staging.rmdir()

    dataset_path = DATASET_DIR.resolve().as_posix()
    yaml_lines = [
        f'path: "{dataset_path}"',
        "train: images/train",
        "val: images/val",
        f"nc: {len(CLASSES)}",
        "names:",
    ]
    yaml_lines.extend(f"  {index}: {name}" for index, name in enumerate(CLASSES))
    (DATASET_DIR / "data.yaml").write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")
    record = {
        "seed": SEED,
        "train_fraction": 0.90,
        "train_images": len(train_pairs),
        "val_images": len(val_pairs),
    }
    SPLIT_MARKER.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
    print("Wrote", len(train_pairs), "train and", len(val_pairs), "val")

TRAIN_COUNT = len(image_files(DATASET_DIR / "images" / "train"))
VAL_COUNT = len(image_files(DATASET_DIR / "images" / "val"))
print(f"On disk: {TRAIN_COUNT} train, {VAL_COUNT} val")


Split already written: 7936 train, 882 val
On disk: 7936 train, 882 val


## 4. YOLOv8 training configuration

Trains YOLOv8s for 40 epochs at 640, patience 10, mosaic 1.0, mixup 0.1, cls 1.5, and box 7.5. Before training, the cell writes the dataset folder into `data.yaml` so the image paths resolve next to that file.

Batch starts at 16. On a GPU with at least 6 GB and compute capability 7 or newer, AMP stays on and the optimizer stays on auto. That is the Kaggle setting. On a smaller GPU the same cell switches to batch 4, AMP off, and SGD, then starts training. The run is saved under `artifacts/yolov8/ppe_yolov8s`.


In [4]:
from ultralytics import YOLO

YOLO_CFG = {
    "model": "yolov8s.pt",
    "epochs": 40,
    "imgsz": 640,
    "batch": 16,
    "patience": 10,
    "mosaic": 1.0,
    "mixup": 0.1,
    "cls": 1.5,
    "box": 7.5,
    "device": DEVICE,
}

YOLO_PROJECT = PROJECT_ROOT / "artifacts" / "yolov8"
YOLO_RUN_NAME = "ppe_yolov8s"
YOLO_RUN_DIR = YOLO_PROJECT / YOLO_RUN_NAME
YOLO_EXPORT = PROJECT_ROOT / "artifacts" / "weights" / "yolov8" / "best.pt"
DATA_YAML = DATASET_DIR / "data.yaml"
dataset_path = DATASET_DIR.resolve().as_posix()
data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
data_cfg["path"] = dataset_path
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding="utf-8")

gpu = torch.cuda.get_device_properties(0) if CUDA else None
small_gpu = gpu is None or gpu.major < 7 or gpu.total_memory < 6 * 1024**3
if small_gpu:
    YOLO_CFG["batch"] = 4
    TRAIN_AMP = False
    TRAIN_OPTIMIZER = "SGD"
else:
    TRAIN_AMP = CUDA
    TRAIN_OPTIMIZER = "auto"

print("data:    ", DATA_YAML)
print("path:    ", dataset_path)
print("project: ", YOLO_PROJECT)
print("device:  ", YOLO_CFG["device"])
print("amp:     ", TRAIN_AMP)
print("optimizer:", TRAIN_OPTIMIZER)
for key in ("epochs", "imgsz", "batch", "patience", "mosaic", "mixup", "cls", "box"):
    print(f"{key}: {YOLO_CFG[key]}")
if small_gpu and CUDA:
    print(
        f"{torch.cuda.get_device_name(0)} has {gpu.total_memory / 1024**3:.1f} GB and compute {gpu.major}.{gpu.minor}. "
        "Batch 16 with AMP does not fit, and the driver resets the GPU."
    )

yolo_model = YOLO(YOLO_CFG["model"])
yolo_model.train(
    data=str(DATA_YAML),
    epochs=YOLO_CFG["epochs"],
    imgsz=YOLO_CFG["imgsz"],
    batch=YOLO_CFG["batch"],
    patience=YOLO_CFG["patience"],
    mosaic=YOLO_CFG["mosaic"],
    mixup=YOLO_CFG["mixup"],
    cls=YOLO_CFG["cls"],
    box=YOLO_CFG["box"],
    device=YOLO_CFG["device"],
    project=str(YOLO_PROJECT),
    name=YOLO_RUN_NAME,
    exist_ok=True,
    seed=SEED,
    pretrained=True,
    plots=True,
    workers=2,
    amp=TRAIN_AMP,
    optimizer=TRAIN_OPTIMIZER,
)


data:     C:\Users\My Pc\Documents\PPE-SofEng\data\processed\PPE-Normalized-v2\data.yaml
path:     C:/Users/My Pc/Documents/PPE-SofEng/data/processed/PPE-Normalized-v2
project:  C:\Users\My Pc\Documents\PPE-SofEng\artifacts\yolov8
device:   0
amp:      False
optimizer: SGD
epochs: 40
imgsz: 640
batch: 4
patience: 10
mosaic: 1.0
mixup: 0.1
cls: 1.5
box: 7.5
NVIDIA GeForce MX130 has 2.0 GB and compute 5.0. Batch 16 with AMP does not fit, and the driver resets the GPU.
New https://pypi.org/project/ultralytics/8.4.184 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.175  Python-3.13.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce MX130, 2048MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\U

KeyboardInterrupt: 

## 5. Testing

Loads `artifacts/yolov8/ppe_yolov8s/weights/best.pt` and evaluates it on the 10% validation split. Image size, batch, and device come from section 4. Plots are written to `artifacts/yolov8/ppe_yolov8s_val`.


In [ ]:
best_src = YOLO_RUN_DIR / "weights" / "best.pt"
if not best_src.is_file():
    raise FileNotFoundError(best_src)

yolo_best = YOLO(str(best_src))
yolo_test = yolo_best.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=YOLO_CFG["imgsz"],
    batch=YOLO_CFG["batch"],
    device=YOLO_CFG["device"],
    plots=True,
    project=str(YOLO_PROJECT),
    name=YOLO_RUN_NAME + "_val",
    exist_ok=True,
)
YOLO_VAL_DIR = YOLO_PROJECT / (YOLO_RUN_NAME + "_val")
print("Validation run:", YOLO_VAL_DIR)


## 6. Model evaluation metrics and confusion matrix

Reads the section 5 validation result. Prints overall precision, recall, mAP50, and mAP50-95, then a per-class table for all seven classes. Shows `confusion_matrix_normalized.png` and `confusion_matrix.png` from the validation folder when those files exist, otherwise from the training run folder.


In [ ]:
box = yolo_test.box
indexes = [int(class_id) for class_id in list(getattr(box, "ap_class_index", []) or [])]
class_ids_found = indexes if indexes and len(indexes) == len(box.p) else list(range(min(len(CLASSES), len(box.p))))
precision = [0.0] * len(CLASSES)
recall = [0.0] * len(CLASSES)
ap50 = [0.0] * len(CLASSES)
ap = [0.0] * len(CLASSES)
for row, class_id in enumerate(class_ids_found):
    precision[class_id] = float(box.p[row])
    recall[class_id] = float(box.r[row])
    ap50[class_id] = float(box.ap50[row])
    ap[class_id] = float(box.ap[row])

yolo_metrics = {
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50-95": float(box.map),
    "per_class": {
        name: {
            "precision": precision[index],
            "recall": recall[index],
            "mAP50": ap50[index],
            "mAP50-95": ap[index],
        }
        for index, name in enumerate(CLASSES)
    },
}
print(
    f"P={yolo_metrics['precision']:.4f}  R={yolo_metrics['recall']:.4f}  "
    f"mAP50={yolo_metrics['mAP50']:.4f}  mAP50-95={yolo_metrics['mAP50-95']:.4f}"
)
display(pd.DataFrame(yolo_metrics["per_class"]).T.round(4))

for folder in (YOLO_VAL_DIR, YOLO_RUN_DIR):
    for name in ("confusion_matrix_normalized.png", "confusion_matrix.png"):
        image = folder / name
        if image.is_file():
            print(image)
            display(ShowImage(filename=str(image)))


## 7. YOLOv8 training artifacts

Copies `best.pt` to `artifacts/weights/yolov8/best.pt` and writes the seven class names beside it as `best.classes.yaml`. Writes `ppe_val_metrics.json` into the training run folder and lists every file in that folder.


In [ ]:
YOLO_EXPORT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_src, YOLO_EXPORT)
class_sidecar = YOLO_EXPORT.with_suffix(".classes.yaml")
class_sidecar.write_text(
    yaml.safe_dump({"nc": len(CLASSES), "names": CLASSES}, sort_keys=False),
    encoding="utf-8",
)
(YOLO_RUN_DIR / "ppe_val_metrics.json").write_text(
    json.dumps(yolo_metrics, indent=2),
    encoding="utf-8",
)

print("Best weights:", YOLO_EXPORT)
print("Class list:  ", class_sidecar)
print("Run folder:  ", YOLO_RUN_DIR)
for path in sorted(YOLO_RUN_DIR.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(YOLO_RUN_DIR).as_posix())


## 8. RF-DETR training and evaluation

Trains RF-DETR Small on the same 90/10 folders for 40 epochs, resolution 640, learning rate 1e-4, and patience 10. Batch size follows section 4. When the GPU has under 6 GB, this cell uses batch 1.

After training it loads `artifacts/rfdetr/ppe_rfdetr_small/checkpoint_best_total.pth`, evaluates the validation split, and writes `ppe_val_metrics.json` with precision, recall, mAP50, and mAP50-95.


In [ ]:
import rfdetr

RF_OUTPUT = PROJECT_ROOT / "artifacts" / "rfdetr" / "ppe_rfdetr_small"
RF_OUTPUT.mkdir(parents=True, exist_ok=True)
RF_CFG = {
    "model": "RFDETRSmall",
    "epochs": YOLO_CFG["epochs"],
    "resolution": YOLO_CFG["imgsz"],
    "batch_size": YOLO_CFG["batch"],
    "patience": YOLO_CFG["patience"],
    "lr": 1.0e-4,
    "device": RF_DEVICE,
}
if CUDA and torch.cuda.get_device_properties(0).total_memory < 6 * 1024**3:
    RF_CFG["batch_size"] = 1
print("dataset: ", DATASET_DIR)
print("output:  ", RF_OUTPUT)
for key, value in RF_CFG.items():
    print(f"{key}: {value}")

rf_model = rfdetr.RFDETRSmall(resolution=RF_CFG["resolution"])
rf_model.train(
    dataset_dir=str(DATASET_DIR),
    dataset_file="yolo",
    class_names=CLASSES,
    epochs=RF_CFG["epochs"],
    batch_size=RF_CFG["batch_size"],
    lr=RF_CFG["lr"],
    resolution=RF_CFG["resolution"],
    device=RF_CFG["device"],
    output_dir=str(RF_OUTPUT),
    early_stopping=True,
    early_stopping_patience=RF_CFG["patience"],
)


def to_python(value):
    if hasattr(value, "item") and getattr(value, "ndim", 0) == 0:
        return float(value.item())
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def first_metric(metrics: dict, *keys: str):
    for key in keys:
        if key in metrics and isinstance(metrics[key], (int, float)):
            return float(metrics[key])
    return None


RF_BEST = RF_OUTPUT / "checkpoint_best_total.pth"
if not RF_BEST.is_file():
    raise FileNotFoundError(RF_BEST)

best_rf = rfdetr.RFDETRSmall(pretrain_weights=str(RF_BEST), resolution=RF_CFG["resolution"])
raw_rf = {
    key: to_python(value)
    for key, value in dict(
        best_rf.evaluate(
            dataset_dir=str(DATASET_DIR),
            dataset_file="yolo",
            split="val",
            device=RF_DEVICE,
        )
    ).items()
}
rf_metrics = {
    "model": "RF-DETR Small",
    "weights": str(RF_BEST),
    "precision": first_metric(raw_rf, "val/precision", "test/precision"),
    "recall": first_metric(raw_rf, "val/recall", "test/recall", "val/mAR", "test/mAR"),
    "mAP50": first_metric(raw_rf, "val/mAP_50", "test/mAP_50", "val/map_50", "test/map_50"),
    "mAP50-95": first_metric(raw_rf, "val/mAP_50_95", "test/mAP_50_95", "val/map", "test/map"),
    "raw_metrics": raw_rf,
}
(RF_OUTPUT / "ppe_val_metrics.json").write_text(
    json.dumps(rf_metrics, indent=2, default=str),
    encoding="utf-8",
)
print("Metric keys:")
for key in sorted(raw_rf):
    print(" ", key)
print(
    f"P={rf_metrics['precision']}  R={rf_metrics['recall']}  "
    f"mAP50={rf_metrics['mAP50']}  mAP50-95={rf_metrics['mAP50-95']}"
)
print("Checkpoint:", RF_BEST)
